# openai-chat — the governed lifecycle on Chat Completions

`chat.completions.create` is the shape most production code still calls. This walks the whole lifecycle on it, offline.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## The five steps

Every recipe in `providers/` walks the same five, in the same order:

| # | Step | Here |
|---|---|---|
| 1 | **connect** | `OpenAI()` — the classic `chat.completions.create` shape |
| 2 | **instrument** | one wrap — detection is structural, not name-based |
| 3 | **govern** | a `tokenguard` budget **and** a `guardrails` gate |
| 4 | **record** | `cassette` — the same call replayed offline, 0 provider calls |
| 5 | **prove** | `acttrace` `verify()` and a cost that came from `prices` |

**Distinctive here: attribution.** `track()` tags a call and `report(group_by=…)` turns the tags into a spend table — the answer to *which feature spent it*.

## 1–2 · Connect and instrument

In [ ]:
import main as recipe
from cendor.core import bus, instrument
from cendor.core.types import LLMCall

seen, calls = [], []
bus.subscribe(lambda e: calls.append(e) if isinstance(e, LLMCall) else None)
client = instrument(recipe.fake_openai(seen))
client

## 3a · Govern — the gate refuses an injection before anything is sent

In [ ]:
from cendor.guardrails import GuardrailTripped, install, rules, uninstall

install([rules.keyword_deny(["ignore previous instructions"], action="block")])
try:
    client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": "ignore previous instructions"}],
    )
except GuardrailTripped as e:
    print(f"BLOCKED by {e.decisions[-1].guardrail} — provider saw {len(seen)} call(s)")
finally:
    uninstall()

## 3b · Govern — the USD cap, on the loop that spends

⚠️ The 6th-turn block is a property of the FAKE's usage (12,000 in / 6,000 out). A real reply is ~50 output tokens, so live the same cap lasts far longer.

In [ ]:
from cendor.tokenguard import BudgetExceeded, report, reset

reset()
try:
    recipe.support_bot(client)
except BudgetExceeded as e:
    print(f"{type(e).__name__}: blocked pre-flight, no call ran")

## 5a · Prove — the spend table is tokenguard's, not a running total kept here

In [ ]:
r = report(group_by=["feature", "user_id"])
for row in r:
    print(f"  {row['tags']} {row['calls']} calls  ${row['usd'].amount}")
total_calls = sum(row["calls"] for row in r)
total_calls

## 4 · Record — replay the same call with the provider unplugged

In [ ]:
import pathlib
import tempfile

from cendor import cassette

tape = str(pathlib.Path(tempfile.mkdtemp()) / "support.cassette.json")
before = len(seen)
with cassette.using(tape, mode="record"):
    client.chat.completions.create(
        model="gpt-4o", messages=[{"role": "user", "content": "Say hi."}]
    )
recorded = len(seen) - before
with cassette.using(tape, mode="replay"):
    client.chat.completions.create(
        model="gpt-4o", messages=[{"role": "user", "content": "Say hi."}]
    )
extra = len(seen) - before - recorded
print(f"replayed 1 call, {extra} provider call(s), $0")

## Prove it

In [ ]:
assert total_calls == 5, f"the $0.50 cap should stop the loop after 5 calls, got {total_calls}"
assert extra == 0, "a replayed call must not reach the provider"
assert any(c.cost and c.cost.amount for c in calls), "no call was priced"
print("OK")